# Evaluation of the hierarchy attribute extraction pipeline

Here we evaluate the performance of **Step 2** of the Ariadne pipeline: extracting SNOMED CT hierarchy attributes for medical terms.

The pipeline:
1. Retrieves similar reference SNOMED terms (few-shot examples)
2. Uses an LLM to extract attribute components from the medical term
3. Retrieves candidate concepts for each extracted component via vector search
4. Uses an LLM to select the best candidate for each attribute

## Setup

Before running this notebook, make sure the environment is set up as described in the README.
This includes:
- Credentials for a database with the OHDSI Vocabulary loaded (including `snomed_attribute` and `snomed_reference` tables)
  If you need to recreate these two tables, please run /sandbox/build_pg_indexes_attributes.py
- Credentials for the LLM API
- A `config.yaml` with the hierarchy prompts configured

All results will be stored in the `data/notebook_results` folder.
Most code blocks will load results from file if they already exist, to save time and costs.
Delete those files to rerun the corresponding steps.

## Configuration

Load the hierarchy pipeline configuration from `config.yaml`.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path.cwd().parent.parent
load_dotenv()

from ariadne.utils.config import load_hierarchy_settings

cfg = load_hierarchy_settings(str(project_root / "config.yaml"))
print(f"Extraction model: {cfg.models.extraction}")
print(f"Selection model: {cfg.models.selection}")
print(f"Reference examples: {cfg.retrieval.num_reference_examples}")
print(f"Top-k per category: {cfg.retrieval.top_k_per_category}")

## Gold standard

We use the hierarchy attributes gold standard file.
Each row maps a source concept to an expected SNOMED attribute (e.g. finding site, associated morphology).

This gold standard contains the following columns:
- `concept_id_1`: the concept ID of the source term
- `concept_name_1`: the source term text
- `concept_id_2`: the concept ID of the expected attribute value (e.g. a finding site concept)
- `concept_code_2`: the concept code of the expected attribute value (e.g. a finding site concept)
- `concept_name_2`: the name of the expected attribute value
- `attribute_category`: the SNOMED relationship type (e.g. `Has finding site`, `Has asso morph`, `Has causative agent`)

In [ ]:
attribute_gs_path = project_root / "data" / "gold_standards" / "hierarchy_attributes_snomed_gs.csv"
# Alternative: use unmatched terms from the exact matching pipeline (Step 1)
# attribute_gs_path = project_root / "data" / "notebook_results" / "exact_matching_vector_search_results.csv"

attribute_gs = pd.read_csv(attribute_gs_path)

# If using exact_matching_vector_search_results.csv, filter to unmatched terms only:
# attribute_gs = attribute_gs[attribute_gs["mapped_concept_id"] == -1]

print(f"Attribute GS: {len(attribute_gs)} rows")
unique_terms = attribute_gs[["concept_id_1", "concept_name_1"]].drop_duplicates()
print(f"Unique terms: {len(unique_terms)}")
attribute_gs.head(10)

## Run the hierarchy attribute extraction pipeline

We use `process_gold_standard` from the evaluator module, which:
1. Iterates over each unique term in the gold standard
2. Runs the four-step attribute extraction pipeline (`find_attributes_two_stage`)
3. Supports checkpointing (resumes from where it left off if interrupted)


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")

from ariadne.hierarchy.evaluator import process_gold_standard
from ariadne.hierarchy.evaluator import _build_prediction_rows
from ariadne.hierarchy.searchers import SnomedAttributeSearcher, SnomedReferenceSearcher

raw_results_file = project_root / "data" / "notebook_results" / "hierarchy_results_raw.json"

if raw_results_file.exists():
    with open(raw_results_file, "r") as f:
        results = json.load(f)
    print(f"Loaded {len(results)} cached results from file.")
else:
    # Exclude gold standard terms from reference examples to prevent data leakage
    gs_concept_ids = set(attribute_gs["concept_id_1"].unique())
    with SnomedAttributeSearcher(cfg=cfg) as attr_idx, \
         SnomedReferenceSearcher(cfg=cfg, exclude_concept_ids=gs_concept_ids) as ref_idx:
        results = process_gold_standard(
            str(attribute_gs_path),
            attr_idx,
            reference_index=ref_idx,
            cfg=cfg,
            max_workers=8,  
        )
    # Save raw results for debugging
    with open(raw_results_file, "w") as f:
        json.dump(results, f, indent=2, default=str)
    print(f"Processed {len(results)} terms. Results saved.")

total_cost = sum(r.get("cost", {}).get("total_cost", 0.0) for r in results if "cost" in r)
print(f"Total API cost: ${total_cost:.4f}")

# Save flat predictions CSV (one row per attribute) — input for RF2 export
results_df = pd.DataFrame(_build_prediction_rows(results))
results_df.to_csv(project_root / "data" / "notebook_results" / "attribute_results.csv", index=False)
print(f"Saved {len(results_df)} attribute rows to attribute_results.csv")
print(f"Columns: {results_df.columns.tolist()}")
results_df.head(10)

## Evaluation

We evaluate the pipeline results against the gold standard using a full outer join.
Each predicted attribute is matched against the expected attribute by `(concept_id_1, concept_id_2, attribute_category)`.
This gives us precision, recall, and F1 scores.

In [ ]:
from ariadne.hierarchy.evaluator import evaluate_results

# Set output directory relative to project root
cfg.evaluation.output_dir = str(project_root / "data" / "notebook_results")

eval_df = evaluate_results(results, str(attribute_gs_path), cfg=cfg)
eval_df.head(20)

### Per-category breakdown

In [ ]:
n_match = (eval_df["status"] == "match").sum()
n_missed = (eval_df["status"] == "missed").sum()
n_extra = (eval_df["status"] == "extra").sum()
n_gs = n_match + n_missed
n_pred = n_match + n_extra

precision = n_match / n_pred * 100 if n_pred else 0.0
recall = n_match / n_gs * 100 if n_gs else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print(f"Gold standard rows: {n_gs}")
print(f"Predicted rows:     {n_pred}")
print(f"Matched:            {n_match}")
print(f"Precision:          {precision:.1f}%")
print(f"Recall:             {recall:.1f}%")
print(f"F1:                 {f1:.1f}%")

breakdown = (
    eval_df.groupby("attribute_category")["status"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=["match", "missed", "extra"], fill_value=0)
)
breakdown["precision%"] = (
    breakdown["match"] / (breakdown["match"] + breakdown["extra"]).clip(lower=1) * 100
).round(1)
breakdown["recall%"] = (
    breakdown["match"] / (breakdown["match"] + breakdown["missed"]).clip(lower=1) * 100
).round(1)

breakdown

### Missed and extra predictions

Inspect which attributes were missed by the pipeline and which were incorrectly predicted.

In [ ]:
missed = eval_df[eval_df["status"] == "missed"][["concept_id_1", "concept_name_1", "attribute_category", "gs_concept_name_2"]]
print(f"Missed predictions: {len(missed)}")

extra = eval_df[eval_df["status"] == "extra"][["concept_id_1", "concept_name_1", "attribute_category", "predicted_concept_name_2"]]
print(f"Extra predictions: {len(extra)}")

joined = missed.merge(
    extra,
    on=["concept_id_1", "concept_name_1", "attribute_category"],
    how="outer",
    suffixes=("_missed", "_extra"),
    indicator=True
)

joined.head(20)


## RF2 Delta Export

Convert the predicted attributes to SNOMED CT RF2 delta files for use with the ELK reasoner.
Source OMOP concept IDs are remapped to synthetic SCTIDs (starting at 1 000 000 001)
to avoid collisions with real SNOMED IDs in the base release.

Concept definitions are expressed as **OWL Functional Syntax axioms** in the OWL Axiom Reference Set,
which gives ELK more precise subsumption semantics than the legacy StatedRelationship format:

- A concept with attributes becomes `EquivalentClasses(:src ObjectIntersectionOf(:parent ObjectSomeValuesFrom(:609096000 ...)))`
- A concept with no attributes falls back to `SubClassOf(:src :parent)`

The output ZIP contains files under `Delta/`:
- `Delta/Terminology/sct2_Concept_Delta_INT_{date}.txt` — one row per source term (sufficiently defined)
- `Delta/Terminology/sct2_StatedRelationship_Delta_INT_{date}.txt` — header-only placeholder
- `Delta/Terminology/sct2_Relationship_Delta_INT_{date}.txt` — header-only placeholder (required by toolkit)
- `Delta/Refset/Content/der2_sRefset_OWLAxiomDelta_INT_{date}.txt` — OWL axiom per source concept
- `Delta/Refset/Metadata/der2_ssRefset_ModuleDependencyDelta_INT_{date}.txt` — module dependency

In [ ]:

# ── Stated parent selection via reference-term neighbourhood voting ────────
import importlib
import json
import psycopg
import ariadne.hierarchy.parent_selector as _ps_mod
importlib.reload(_ps_mod)
from ariadne.hierarchy.parent_selector import build_stated_parents_map
from ariadne.utils.utils import get_environment_variable

conn_str = get_environment_variable("VOCAB_CONNECTION_STRING")
conn_str = conn_str.replace("+psycopg", "").replace("+psycopg2", "")
schema = get_environment_variable("VOCAB_SCHEMA")


# Load raw hierarchy results (contains reference_examples with concept_id + similarity)
raw_results_path = project_root / "data" / "notebook_results" / "hierarchy_results_raw.json"
with open(raw_results_path) as fh:
    raw_results = json.load(fh)

print(f"Loaded {len(raw_results)} raw result entries.")

# ── Tuning parameters ─────────────────────────────────────────────────
#   top_k=1: single best parent (reduces ancestor-explosion false positives)
#   min_similarity=0.7: minimum cosine similarity to count a reference term
#   use_attr_filter=True: skip reference terms less specific than the source
PARENT_TOP_K = 2
PARENT_MIN_SIM = 0.6

with psycopg.connect(conn_str) as parent_conn:
    stated_parents_map = build_stated_parents_map(
        raw_results,
        parent_conn,
        schema,
        top_k=PARENT_TOP_K,
        min_similarity=PARENT_MIN_SIM,
        use_attr_filter=True,
    )

# Coverage report
fallback = [v for v in stated_parents_map.values() if v == ["404684003"]]
specific = len(stated_parents_map) - len(fallback)
print(f"\nStated parent coverage (top_k={PARENT_TOP_K}, min_sim={PARENT_MIN_SIM}):")
print(f"  Concepts with specific parent(s) : {specific}/{len(stated_parents_map)}")
print(f"  Fell back to Clinical finding    : {len(fallback)}/{len(stated_parents_map)}")

# Sample 5 concepts with their selected parents
sample_ids = list(stated_parents_map)[:5]
for cid in sample_ids:
    src_name = next((e.get("source_concept_name","?") for e in raw_results
                     if e.get("source_concept_id") == cid), "?")
    print(f"  {src_name!r:50s} → {stated_parents_map[cid]}")


In [ ]:
import importlib, sys, shutil

# Force a fresh import — drop ALL ariadne.hierarchy modules from cache
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("ariadne"):
        del sys.modules[mod_name]

from ariadne.hierarchy.rf2_exporter import export_to_rf2

rf2_output_dir = project_root / "data" / "rf2_output"
# Clean previous delta files to avoid stale artifacts
old_delta = rf2_output_dir / "SnomedCT_YourDelta"
if old_delta.exists():
    shutil.rmtree(old_delta)
old_delta2 = rf2_output_dir / "Delta"
if old_delta2.exists():
    shutil.rmtree(old_delta2)

zip_path, id_map = export_to_rf2(
    source=project_root / "data" / "notebook_results" / "attribute_results.csv",
    output_dir=rf2_output_dir,
    stated_parent=stated_parents_map,   # per-concept parents from neighbourhood voting
)
print(f"RF2 delta written to: {zip_path}")
print(f"ID mapping: {len(id_map)} concepts (OMOP → synthetic)")

# Verify OWL content inline
import zipfile
with zipfile.ZipFile(zip_path) as zf:
    owl_name = next(n for n in zf.namelist() if "OWLAxiom" in n)
    with zf.open(owl_name) as f:
        owl_lines = f.read().decode("utf-8").splitlines()
print(f"OWL axiom rows: {len(owl_lines) - 1}")
if owl_lines[1:]:
    sample = owl_lines[1].split("\t")
    print(f"Sample expression: {sample[-1][:150]}")
id_map.head()


## SNOMED Classification (ELK Reasoner)

Run the ELK OWL EL++ classifier (via [snomed-owl-toolkit](https://github.com/IHTSDO/snomed-owl-toolkit))
to infer **Is a** (parent) relationships from the predicted attributes.

The classifier takes the RF2 delta ZIP produced above and a base SNOMED CT International Edition
snapshot, converts both to OWL, runs ELK, and returns the inferred parent relationships.

### Prerequisites
- **Java 17+** installed and on PATH
- **snomed-owl-toolkit JAR** — download from [GitHub releases](https://github.com/IHTSDO/snomed-owl-toolkit/releases)
- **SNOMED CT International Edition RF2 snapshot** — obtain from [MLDS](https://mlds.ihtsdotools.org/)

### Pre-classification validation

Check the RF2 delta for structural issues before sending it to the classifier.

In [ ]:
from ariadne.hierarchy.classifier import (
    classify_delta,
    classification_summary,
    parse_classification_results,
    pre_classification_checks,
    resolve_parent_names,
)

delta_zip = project_root / "data" / "rf2_output" / next(
    f.name for f in (project_root / "data" / "rf2_output").iterdir()
    if f.name.startswith("snomed_delta_") and f.name.endswith(".zip")
)
print(f"Delta ZIP: {delta_zip}")

issues = pre_classification_checks(delta_zip)
if issues:
    print("⚠️  Issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("✅ All pre-classification checks passed.")

### Run classification

This step takes ~90-120 seconds (dominated by loading the base SNOMED release).
The snomed-owl-toolkit converts RF2 → OWL, runs ELK, and produces a results ZIP
with the inferred "Is a" relationships.

In [ ]:
results_zip = classify_delta(
    delta_zip,
    base_snomed_zip=str(project_root / "data" / "SnomedCT_InternationalRF2.zip"),
    toolkit_jar=str(project_root / "tools" / "snomed-owl-toolkit-5.3.0-executable.jar"),
    output_dir=project_root / "data" / "rf2_output",
)
print(f"Classification results: {results_zip}")

### Parse results and resolve names

Extract the new inferred "Is a" relationships and resolve SNOMED concept names.

In [ ]:

new_is_a, removed, equiv_df = parse_classification_results(results_zip)

print(f"New inferred 'Is a' relationships: {len(new_is_a)}")
print(f"Redundant relationships removed:   {len(removed)}")
print(f"Equivalent concept rows:           {len(equiv_df)}")

# Build lookup structures
synth_to_omop = dict(zip(
    id_map["synthetic_sctid"].astype(str),
    id_map["omop_concept_id"].astype(str),
))
synth_ids = set(id_map["synthetic_sctid"].astype(str))

# src_name_map: OMOP ID → concept name (from id_map + attribute_gs override)
src_name_map = {}
for _, m in id_map.iterrows():
    omop_str = str(m["omop_concept_id"])
    if str(m.get("concept_name", "")):
        src_name_map[omop_str] = str(m["concept_name"])
for _, gs_row in attribute_gs.iterrows():
    src_name_map[str(gs_row["concept_id_1"])] = gs_row["concept_name_1"]

# Collect all real SCTIDs that appear as ELK results (parents or children)
real_sctids = set()
for _, row in new_is_a.iterrows():
    src, dest = str(row["sourceId"]), str(row["destinationId"])
    src_is_synth = src in synth_ids
    dest_is_synth = dest in synth_ids
    if src_is_synth and not dest_is_synth:
        real_sctids.add(dest)
    elif not src_is_synth and dest_is_synth:
        real_sctids.add(src)

# Resolve SCTID → concept_name for display (keep SCTID as the ID)
sctid_name_map: dict[str, str] = {}
if real_sctids:
    with psycopg.connect(conn_str) as conn:
        with conn.cursor() as cur:
            cur.execute(
                f"""
                SELECT concept_code, concept_name
                FROM {schema}.concept
                WHERE vocabulary_id = 'SNOMED'
                  AND concept_code = ANY(%s)
                """,
                (list(real_sctids),),
            )
            for code, name in cur.fetchall():
                sctid_name_map[str(code)] = name

unmapped = real_sctids - set(sctid_name_map)
if unmapped:
    print(f"{len(unmapped)} SCTIDs not found in CONCEPT table (will keep SCTID as fallback)")

# Build equivalence pairs from the ELK equivalences refset
# Group by mapTarget UUID — concepts sharing the same group are equivalent
from collections import defaultdict
equiv_groups: dict[str, list[str]] = defaultdict(list)
for _, row in equiv_df.iterrows():
    sctid = str(row["referencedComponentId"])
    group = str(row["mapTarget"])
    equiv_groups[group].append(sctid)

# Generate "Maps to" pairs from equivalence groups (all pairwise within group)
equiv_pairs_set: set[tuple[str, str]] = set()
for group_members in equiv_groups.values():
    # Only keep pairs where at least one member is a synthetic (delta) concept
    synth_members = [m for m in group_members if m in synth_ids]
    if not synth_members:
        continue
    for i, a in enumerate(group_members):
        for b in group_members[i+1:]:
            if a in synth_ids or b in synth_ids:
                equiv_pairs_set.add((min(a, b), max(a, b)))

print(f"Equivalence groups: {len(equiv_groups)}, pairs involving delta concepts: {len(equiv_pairs_set)}")

# Build parents_df
rows = []
skipped_base_only = 0
for _, row in new_is_a.iterrows():
    src, dest = str(row["sourceId"]), str(row["destinationId"])
    src_is_synth = src in synth_ids
    dest_is_synth = dest in synth_ids

    if src_is_synth and dest_is_synth:
        continue  # inter-delta hierarchy handled via equivalences
    if not src_is_synth and not dest_is_synth:
        skipped_base_only += 1
        continue

    if src_is_synth and not dest_is_synth:
        omop_id = synth_to_omop[src]
        rows.append({
            "concept_id_1":   omop_id,
            "concept_name_1": src_name_map.get(omop_id, omop_id),
            "relationship_id": "Is a",
            "concept_code_2": dest,
            "concept_name_2": sctid_name_map.get(dest, dest),
        })
    else:
        omop_id = synth_to_omop[dest]
        rows.append({
            "concept_id_1":   omop_id,
            "concept_name_1": src_name_map.get(omop_id, omop_id),
            "relationship_id": "Subsumes",
            "concept_code_2": src,
            "concept_name_2": sctid_name_map.get(src, src),
        })

# Add equivalent pairs as "Maps to"
for a, b in equiv_pairs_set:
    a_is_synth = a in synth_ids
    b_is_synth = b in synth_ids
    if a_is_synth and b_is_synth:
        # Both are delta concepts → use OMOP IDs
        omop_a = synth_to_omop[a]
        omop_b = synth_to_omop[b]
        rows.append({
            "concept_id_1":   omop_a,
            "concept_name_1": src_name_map.get(omop_a, omop_a),
            "relationship_id": "Maps to",
            "concept_code_2": omop_b,
            "concept_name_2": src_name_map.get(omop_b, omop_b),
        })
    elif a_is_synth:
        # a is delta, b is real SNOMED
        omop_a = synth_to_omop[a]
        rows.append({
            "concept_id_1":   omop_a,
            "concept_name_1": src_name_map.get(omop_a, omop_a),
            "relationship_id": "Maps to",
            "concept_code_2": b,
            "concept_name_2": sctid_name_map.get(b, b),
        })
    else:
        # b is delta, a is real SNOMED
        omop_b = synth_to_omop[b]
        rows.append({
            "concept_id_1":   omop_b,
            "concept_name_1": src_name_map.get(omop_b, omop_b),
            "relationship_id": "Maps to",
            "concept_code_2": a,
            "concept_name_2": sctid_name_map.get(a, a),
        })

parents_df = pd.DataFrame(rows, columns=[
    "concept_id_1", "concept_name_1", "relationship_id",
    "concept_code_2", "concept_name_2",
])

parents_csv = project_root / "data" / "notebook_results" / "classification_parents.csv"
parents_df.to_csv(parents_csv, index=False)

n_maps_to = (parents_df["relationship_id"] == "Maps to").sum()
print(f"Saved {len(parents_df)} parent relationships → classification_parents.csv")
print(f"  'Is a'     : {(parents_df['relationship_id']=='Is a').sum()}")
print(f"  'Subsumes' : {(parents_df['relationship_id']=='Subsumes').sum()}")
print(f"  'Maps to'  : {n_maps_to}")
parents_df.head(20)


## Evaluation: inferred parents vs parent gold standard

Compare the ELK-inferred "Is a" parents against the actual direct "Is a" relationships
from the parent gold standard (`hierarchy_snomed_gs.csv`)

In [ ]:
# ── Evaluation: concept-level breakdown ────────────────────────────────────
# For each source concept, determine the outcome:
#   - "Maps to":  equivalent to an existing SNOMED concept (exact match via ELK)
#   - "Is a":     correctly placed under a GS parent (concept_code_2 matches GS)
#   - "Subsumes": correctly subsumes a real SNOMED concept (concept_code_2 matches GS)

parent_gs_path = project_root / "data" / "gold_standards" / "hierarchy_snomed_gs.csv"
parent_gs = pd.read_csv(parent_gs_path, dtype={"concept_code_2": str})

# Ensure consistent types for join keys
parent_gs["concept_id_1"] = parent_gs["concept_id_1"].astype(str)
parent_gs["concept_code_2"] = parent_gs["concept_code_2"].astype(str)

source_ids = set(str(s) for s in attribute_gs["concept_id_1"].unique())
total_source = len(source_ids)

# GS set: (concept_id_1, concept_code_2)
actual_gs = parent_gs[parent_gs["concept_id_1"].isin(source_ids)]
actual_set = {
    (r["concept_id_1"], r["concept_code_2"])
    for _, r in actual_gs.iterrows()
}
print(f"Actual 'Is a' pairs from GS : {len(actual_set)}")

# ── Ensure parents_df keys are strings too ────────────────────────────────
parents_df["concept_id_1"] = parents_df["concept_id_1"].astype(str)
parents_df["concept_code_2"] = parents_df["concept_code_2"].astype(str)

# ── Per-relationship-type sets of source concepts ─────────────────────────
# Concepts with "Maps to" (equivalent to existing SNOMED)
maps_to_df = parents_df[parents_df["relationship_id"] == "Maps to"]
maps_to_concepts = set(maps_to_df["concept_id_1"]) & source_ids

# Concepts with "Is a" where predicted parent matches GS
is_a_df = parents_df[parents_df["relationship_id"] == "Is a"]
predicted_is_a = {
    (row["concept_id_1"], row["concept_code_2"])
    for _, row in is_a_df.iterrows()
}
is_a_correct = predicted_is_a & actual_set
is_a_correct_concepts = {sid for sid, _ in is_a_correct}

# Concepts with "Subsumes"
subsumes_df = parents_df[parents_df["relationship_id"] == "Subsumes"]
subsumes_concepts = set(subsumes_df["concept_id_1"]) & source_ids

# All predicted "Is a" source concepts (regardless of correctness)
is_a_all_concepts = set(is_a_df["concept_id_1"]) & source_ids

# ── Print concept-level summary ───────────────────────────────────────────
print(f"\n{'Category':<45s} {'Count':>6s} {'%':>7s}")
print("-" * 60)
print(f"{'Total source concepts':<45s} {total_source:>6d} {'100.0%':>7s}")
print(f"{'Maps to (equivalent to existing SNOMED)':<45s} {len(maps_to_concepts):>6d} {len(maps_to_concepts)/total_source*100:>6.1f}%")
print(f"{'Is a — matches GS parent (same code)':<45s} {len(is_a_correct_concepts):>6d} {len(is_a_correct_concepts)/total_source*100:>6.1f}%")
print(f"{'Is a — any predicted parent':<45s} {len(is_a_all_concepts):>6d} {len(is_a_all_concepts)/total_source*100:>6.1f}%")
print(f"{'Subsumes (subsumes a real SNOMED concept)':<45s} {len(subsumes_concepts):>6d} {len(subsumes_concepts)/total_source*100:>6.1f}%")
